# Factor Optimization

This notebook evaluates different factor weight combinations and compares strategy performance.

Factors:
- Momentum
- Volatility
- Size

Goal:
Identify the factor weight combination with the highest Sharpe Ratio.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [2]:
data = yf.download(
    "AAPL",
    start="2023-01-01",
    end="2024-01-01"
)

data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2023-01-03,122.982712,128.715409,122.097730,128.105761,112117500
2023-01-04,124.251190,126.512809,122.992553,124.772344,89113600
2023-01-05,122.933533,125.637638,122.677877,125.008319,80962700
2023-01-06,127.456795,128.115611,122.805737,123.907048,87754700
2023-01-09,127.977943,131.183547,127.722288,128.292610,70790800


## Build Factors

In [3]:
# Momentum

data["Momentum_20"] = data["Close"].pct_change(20)

# Volatility

data["Volatility_20"] = (
    data["Close"]
    .pct_change()
    .rolling(20)
    .std()
)

# Size

data["Size_Factor"] = np.log(data["Volume"])

data[
    ["Momentum_20",
     "Volatility_20",
     "Size_Factor"]
].tail()

Price,Momentum_20,Volatility_20,Size_Factor
Ticker,,,
Date,,,
2023-12-22,0.019108,0.009071,17.430464
2023-12-26,0.017177,0.009103,17.180020
2023-12-27,0.014443,0.009086,17.688537
2023-12-28,0.022232,0.008973,17.343338
2023-12-29,0.013582,0.009078,17.569056


## Standardize Factors

In [4]:
data["Momentum_Z"] = (
    data["Momentum_20"]
    - data["Momentum_20"].mean()
) / data["Momentum_20"].std()

data["Volatility_Z"] = (
    data["Volatility_20"]
    - data["Volatility_20"].mean()
) / data["Volatility_20"].std()

data["Size_Z"] = (
    data["Size_Factor"]
    - data["Size_Factor"].mean()
) / data["Size_Factor"].std()

data[
    ["Momentum_Z",
     "Volatility_Z",
     "Size_Z"]
].tail()

Price,Momentum_Z,Volatility_Z,Size_Z
Ticker,,,
Date,,,
2023-12-22,-0.256931,-1.191678,-1.616583
2023-12-26,-0.290610,-1.180316,-2.560388
2023-12-27,-0.338280,-1.186265,-0.644025
2023-12-28,-0.202466,-1.227598,-1.944919
2023-12-29,-0.353294,-1.189463,-1.094293


## Test Weight Combinations

In [34]:
weights = [
    (0.6, 0.2, 0.2),
    (0.5, 0.3, 0.2),
    (0.4, 0.3, 0.3)
]

results = []

for m, v, s in weights:

    score = (
        m * data["Momentum_Z"]
        - v * data["Volatility_Z"]
        + s * data["Size_Z"]
    )

    signal = np.where(score > 0, 1, 0)

    market_returns = data["Close"].squeeze().pct_change().fillna(0)

    returns = (
        pd.Series(signal, index=data.index)
        .shift(1)
        * market_returns
)

    sharpe = (
        returns.dropna().mean()
        / returns.dropna().std()
    ) * np.sqrt(252)

    results.append(
        [m, v, s, sharpe]
    )

results_df = pd.DataFrame(
    results,
    columns=[
        "Momentum",
        "Volatility",
        "Size",
        "Sharpe"
    ]
)

results_df

,Momentum,Volatility,Size,Sharpe
0,0.6,0.2,0.2,1.493950
1,0.5,0.3,0.2,0.468652
2,0.4,0.3,0.3,1.030870


In [36]:
type(market_returns)


pandas.core.series.Series

In [8]:
market_returns.head()

Ticker,AAPL
Date,
2023-01-03,0.000000
2023-01-04,0.010314
2023-01-05,-0.010605
2023-01-06,0.036794
2023-01-09,0.004089


In [12]:
%whos

Variable         Type         Data/Info
---------------------------------------
data             DataFrame    Price            Close   <...>\n[250 rows x 11 columns]
m                float        0.4
market_returns   Series       Date\n2023-01-03    0.000<...>ngth: 250, dtype: float64
np               module       <module 'numpy' from 'c:\<...>ges\\numpy\\__init__.py'>
pd               module       <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>
plt              module       <module 'matplotlib.pyplo<...>\\matplotlib\\pyplot.py'>
results          list         n=3
results_df       DataFrame       Momentum  Volatility  <...>2023-01-03 00:00:00 ...  
returns          DataFrame                AAPL  2023-01<...>n[250 rows x 251 columns]
s                float        0.3
score            Series       Date\n2023-01-03         <...>ngth: 250, dtype: float64
sharpe           Series       AAPL                  NaN<...>ngth: 251, dtype: float64
signal           ndarray      250: 250 elems

In [37]:
type(sharpe)

numpy.float64

In [39]:
results_df

,Momentum,Volatility,Size,Sharpe
0,0.6,0.2,0.2,1.493950
1,0.5,0.3,0.2,0.468652
2,0.4,0.3,0.3,1.030870


## Interpretation

The weight combination (Momentum=0.6, Volatility=0.2, Size=0.2) achieved the highest Sharpe Ratio of 1.494, indicating the best risk-adjusted performance among the tested portfolios. This suggests that momentum is the most important factor in the multi-factor strategy.

In [40]:
results_df.sort_values("Sharpe", ascending=False)

,Momentum,Volatility,Size,Sharpe
0,0.6,0.2,0.2,1.493950
2,0.4,0.3,0.3,1.030870
1,0.5,0.3,0.2,0.468652


## Conclusion

Among the tested factor weight combinations, the portfolio with Momentum (0.6), Volatility (0.2), and Size (0.2) achieved the highest Sharpe Ratio of 1.494. This result suggests that the momentum factor contributed the most to portfolio performance, while volatility and size provided additional diversification benefits.

Overall, the factor optimization process improved the strategy's risk-adjusted returns and identified the most effective factor allocation for this dataset.